In [1]:
import moabb
from moabb.datasets import *
from moabb.evaluations import WithinSessionEvaluation
from moabb.paradigms import P300
from hoda import HODA
from sklearn.pipeline import make_pipeline
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from mne.decoding import Scaler
import matplotlib.pyplot as plt
import seaborn as sns
from moabb.analysis.plotting import paired_plot, meta_analysis_plot, summary_plot
from sklearn.base import BaseEstimator, ClassifierMixin
from toeplitzlda.classification import ToeplitzLDA
from sklearn.svm import SVC
from moabb.analysis.meta_analysis import (  # noqa: E501
    compute_dataset_statistics,
    find_significant_differences,
)

In [2]:
datasets = [
    BNCI2014008(),
    #BNCI2014009()
]

In [3]:
paradigm_1 = P300(resample=64, tmin=0, tmax=1, fmin=0.1, fmax=32)

evaluation_1 = WithinSessionEvaluation(
    paradigm=paradigm_1, datasets=datasets,
    suffix="examples", overwrite=True,
    data_size=dict(policy='ratio', value=[0.1]), n_perms=[5],
)

pipelines_1 = dict()

pipelines_1['tHODA+LDA_64Hz'] = pipeline_1 = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    HODA(max_iter=200, tol=1e-3, rank=(2,2), taper=None,
         toeplitz=(1,), shrinkage=(0.1,0.1), solver='eig'),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'),
)

In [5]:
paradigm_2 = P300(resample=32, tmin=0, tmax=1, fmin=0.1, fmax=16)

evaluation_2 = WithinSessionEvaluation(
    paradigm=paradigm_1, datasets=datasets,
    suffix="examples", overwrite=True,
    data_size=dict(policy='ratio', value=[0.1]), n_perms=[5],
)

pipelines_2 = dict()

pipelines_2['tHODA+LDA_32Hz'] = make_pipeline(
    Scaler(scalings='mean', with_mean=False),
    HODA(max_iter=200, tol=1e-3, rank=(2,2), taper=None,
         toeplitz=(1,), shrinkage=(0.1,0.1), solver='eig'),
    LinearDiscriminantAnalysis(shrinkage='auto', solver='lsqr'),
)

In [6]:
results_1 = evaluation_1.process(pipelines_1)

008-2014-WithinSession:   0%|                                                                   | 0/8 [00:29<?, ?it/s]

KeyboardInterrupt



In [ ]:
results_2 = evaluation_2.process(pipelines_2)

In [ ]:
results = results_1.append(results_2, ignore_index=True)

In [ ]:
stats = compute_dataset_statistics(results)
P, T = find_significant_differences(stats)
_ = summary_plot(P, T)

In [ ]:
_=meta_analysis_plot(stats, "tHODA+LDA_64Hz", "tHODA+LDA_32Hz")

In [ ]:
_=paired_plot(results, "tHODA+LDA_64Hz", "tHODA+LDA_32Hz")